# 真实资产接入与数字孪生

本 Demo 按功能逐项调用 PhysReal-X SDK，不使用隐藏的总流程函数。每个单元格都可单独阅读 API、输入、输出与阻断条件。


## 0. 初始化平台

只在这里定义工作区与输出目录；方法名和参数直接写在对应 API 调用中。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from examples._common import operation_receipt, output_artifact, write_summary
from physrealx import PhysRealX
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / "examples/real_assets_and_digital_twin/output/workspace"
OUTPUT_DIR = PROJECT_ROOT / "examples/real_assets_and_digital_twin/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}
print(f"工作区: {WORKSPACE}")

## 1. 数据接入检查：`assets.inspect_source`

**用途**：在复制数据之前检查 ArtModeling 目录、RGB-D 序列、帧数、格式和缺失项。

**输出**：`SourceInspection`；只有 `ingestible=True` 才能进入导入阶段。


In [ ]:
inspection = px.assets.inspect_source(
    PROJECT_ROOT / "examples/real_assets_and_digital_twin/input/ArtModeling/cabinet2",
    importer="artmodeling",
)
inspection_view = inspection.model_dump(mode="json")
receipts["source_inspection"] = inspection_view
print(json.dumps(inspection_view, ensure_ascii=False, indent=2))

## 2. 发布观测集：`assets.ingest`

**用途**：把通过检查的已采集数据规范化为 `ObservationSetManifest`。

**输出**：版本化 `VersionedRef`；同一资源版本已存在时直接复用，不重复导入。


In [ ]:
observations = next(
    (
        ref
        for ref in px.assets.list_observations()
        if ref.resource_id == "cabinet2.observations" and ref.version == "1.0.0"
    ),
    None,
)
if observations is None and inspection.ingestible:
    observations = px.assets.ingest(
        PROJECT_ROOT / "examples/real_assets_and_digital_twin/input/ArtModeling/cabinet2",
        resource_id="cabinet2.observations",
        version="1.0.0",
        display_name="Cabinet 2 observations",
        importer="artmodeling",
        config={
            "depth_unit": "native_u16_unscaled",
            "coordinate_frame": "camera_unregistered",
            "include_candidates": False,
        },
    )
if observations is None:
    print("[BLOCKED] 数据源未通过接入检查。")
else:
    observation_manifest = px.assets.get_observations(observations)
    receipts["observations"] = observations.model_dump(mode="json")
    print(observations)
    print(f"流数: {len(observation_manifest.streams)}")

## 3. 相机位姿与稀疏重建：`assets.recover_cameras`

**用途**：使用配置的 `method`（默认 COLMAP）恢复相机内参、外参和稀疏点。

**质量指标**：注册率、轨迹长度和重投影误差；单目尺度仍为未解析状态。


In [ ]:
camera_operation = None
camera_rig = None
if observations is None:
    print("[BLOCKED] 需要先发布观测集。")
else:
    try:
        camera_operation = px.assets.recover_cameras(
            observations,
            method="colmap",
            config={
                "execution_device": "auto",
                "allow_cpu_fallback": True,
                "max_frames_per_stream": 32,
                "num_threads": 1,
                "random_seed": 0,
            },
        )
        camera_rig = output_artifact(camera_operation, "camera_rig_calibration")
        receipts["camera_recovery"] = operation_receipt(camera_operation)
        print(json.dumps(camera_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 米制坐标对齐：`assets.align_cameras_metric`

**用途**：用 Marker、标尺或实测三维对应解析 COLMAP 的 Sim(3) 尺度和对象坐标系。

**阻断条件**：不允许从待评估结果猜测尺度；`metric_alignment_evidence` 必须是独立实测证据。


In [ ]:
metric_alignment_evidence = None  # 替换为已发布的 MetricAlignmentEvidence/1 ArtifactRef
metric_operation = None
if camera_rig is None or metric_alignment_evidence is None:
    print("[BLOCKED] 需要相机恢复结果和独立米制对齐证据。")
else:
    try:
        metric_operation = px.assets.align_cameras_metric(
            camera_rig,
            metric_alignment_evidence,
            method="metric_alignment",
            config={},
        )
        receipts["metric_alignment"] = operation_receipt(metric_operation)
        print(json.dumps(metric_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 深度增强：`assets.enhance_depth`

**用途**：对已标定、已对齐到米制坐标的 RGB-D 资源进行深度补全。

**质量门禁**：覆盖率、MAE/RMSE/REL 和边界误差必须通过留出像素评估。


In [ ]:
calibrated_rgbd = None  # 替换为已发布的 CalibratedRGBDBundle/1 VersionedRef
depth_operation = None
if calibrated_rgbd is None:
    print("[BLOCKED] 请在本单元格绑定 CalibratedRGBDBundle/1。")
else:
    try:
        depth_operation = px.assets.enhance_depth(calibrated_rgbd, method="omni_dc", config={})
        receipts["depth_enhancement"] = operation_receipt(depth_operation)
        print(json.dumps(depth_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 6. 视觉几何重建：`assets.reconstruct_geometry`

**用途**：将校验后的深度/点云/多视图束交给选定 `method`，生成按 link 组织的 Mesh、Gaussian、纹理或语义几何。

**输出**：`GeometryReconstructionReport/1` 和视觉几何制品；示例默认选择 `artgs`。


In [ ]:
geometry_source = None  # 替换为已校验的多视图/深度几何输入 ArtifactRef
geometry_operation = None
if geometry_source is None:
    print("[BLOCKED] 请在本单元格绑定几何输入制品。")
else:
    try:
        geometry_operation = px.assets.reconstruct_geometry(
            geometry_source,
            method="artgs",
            config={"mode": "articulation"},
        )
        receipts["geometry_reconstruction"] = operation_receipt(geometry_operation)
        print(json.dumps(geometry_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 7. 部件与关节恢复：`assets.articulate`

**用途**：从同一米制坐标系中的交互前/后点云恢复运动部件、关节类型、轴和原点。

**输出**：`ArticulationModel/1`；若运动量不足或轴有二义性，方法必须拒绝发布。


In [ ]:
paired_pointclouds = None  # 替换为 PairedInteractionPointCloudBundle/1 VersionedRef
articulation_operation = None
if paired_pointclouds is None:
    print("[BLOCKED] 请在本单元格绑定交互前/后点云。")
else:
    try:
        articulation_operation = px.assets.articulate(
            paired_pointclouds,
            method="ditto",
            config={},
        )
        receipts["articulation_recovery"] = operation_receipt(articulation_operation)
        print(json.dumps(articulation_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 8. 保存本次 Demo 回执

回执只记录本次真实完成的调用；未提供证据或未安装方法的阶段保持 `BLOCKED`，不会伪造结果。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "digital-twin-report.json",
    {"module": "真实资产接入与数字孪生", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))